# Main development notebook

In [ ]:
#%%writefile __init__.py
__name__ = ['liboqs_bench']
__version__ = '0.1.0'

# Capture warnings and and disable logging from oqs library
import logging
logging.captureWarnings(True)
logging.getLogger('oqs.oqs').disabled = True

from .liboqs_bench import liboqs_bench

In [125]:
%%writefile custom_exceptions.py
'''
Custom exceptions for liboqs_bench package.
'''
import logging
_logger = logging.getLogger('__main__.' + __name__)

from oqs import oqs

class UnsupportedAlgorithmError(Exception):

    '''Custom exception for the user choosing an unsupported Algorithm or none at all.'''
    
    def __init__(self, alg)-> None:
        alg_str = '"KeyGeneration", "Encapsulation", "Decapsulation", or "all"'
        message = f'''
        "{alg}" is not a supported algorithm or argument.
        
        Type as argument(s):
        - "help" for general support on how to use the tool,
        - "schemes" for a list of supported KEM schemes, or
        - at least two arguments: 
            - One of the KEM algorithms {alg_str} and
            - at least one KEM scheme or "all".

        Optionally for the number of measurement iterations an integer can be passed as keyword argument 
        "iterations". The default value is 1000.
        '''
        super().__init__(message)

class UnsupportedSchemeError(Exception):
    
    '''Custom exception for user choosing unsupported scheme.'''
    
    def __init__(self, scheme:str)-> None:
        schemes_str = ''
        for scheme_name in oqs.get_enabled_kem_mechanisms():
            schemes_str += f'\n"{scheme_name}"'
        message = f'"{scheme}" is not a supported KEM scheme.\n\nInstead use one of these:\n{schemes_str}'
        super().__init__(message)

class NoSchemeProvidedError(Exception):

    '''Custom exception in case user doesn't provide a scheme.'''

    def __init__(self)-> None:
        schemes_str = ''
        for scheme_name in oqs.get_enabled_kem_mechanisms():
            schemes_str += f'\n"{scheme_name}"'
        message = f'No KEM scheme provided.\n\nUse at least one of these:\n{schemes_str}'
        super().__init__(message)
        

Overwriting custom_exceptions.py


In [162]:
#%%writefile schemes.py
'''
KEM schemes for liboqs_bench package
'''

import logging
_logger = logging.getLogger('__main__.' + __name__)

from oqs import oqs

class KeyGeneration(oqs.KeyEncapsulation):

    '''
    Key Generation class
    
    Simple child class to generate a KEM keypair with the chosen scheme. It only exists so that the code is structured. 
    '''
    
    def to_measure(self):
        self.generate_keypair()            

class Encapsulation:

    '''
    Encapsulation class

    The constructor generates two Key Encapsulation instances of Alice and Bob, and generates Alice's keypair.

    The function to_measure() encapsulates a secret in a cipher text with Alice's public key.
    '''
    
    def __init__(self, scheme: str)-> None:
        self.alice = oqs.KeyEncapsulation(scheme)
        self.bob = oqs.KeyEncapsulation(scheme)
        self.alice_pub = self.alice.generate_keypair()
        
    def to_measure(self)-> tuple[bytes, bytes]:        
        return self.bob.encap_secret(self.alice_pub)

class Decapsulation:
    
    '''
    Decapsulation class

    The constructor generates an instance with Alice and her key pair, the shared cipher and secret as Bob sends them.

    The function to_measure() returns the secret as Alice decrypts them.

    The secret as Bob sends it and as Alice receives it can be called for testing reasons.
    '''
    
    def __init__(self, scheme: str)-> None:
        self.alice = oqs.KeyEncapsulation(scheme)
        self.bob = oqs.KeyEncapsulation(scheme)
        self.alice_pub = self.alice.generate_keypair()
        self.cipher,self.bob_secret = self.bob.encap_secret(self.alice_pub)
    
    def to_measure(self) -> bytes:
        return self.alice.decap_secret(self.cipher)

In [177]:
%%writefile liboqs_bench.py
'''

Small Python utility benchmarking PQC KEM algorithms and \
schemes from the liboqs-python library. The program measures each \
algorithm step separately for as many schemes as required by running \
the step a specified number of iterations for five trials in total. \
It then chooses the fastest trial and calculates the average.

Main module containing benchmark classes for algorithms and schemes as well as interactive function.
'''
import logging
_logger = logging.getLogger('__main__.' + __name__)

from oqs import oqs
import timeit
import time
from .custom_exceptions import *
from .schemes import *


class _Benchmark(timeit.Timer):

    '''
    _Benchmark class

    Constructor generates a benchmark instance from the timeit builtin and function benchmark() runs it five \
seperate times for a specified number of iterations.

    '''

    def __init__(self, alg:str, scheme:str):
        self.scheme = scheme
        super().__init__(stmt="bench.to_measure()", setup=f'bench = {alg}("{self.scheme}")', timer=time.perf_counter_ns,
                         globals=globals())
        
    def benchmark(self, iterations):
        try:
            return self.repeat(number=iterations)
        except oqs.MechanismNotSupportedError:
            raise UnsupportedSchemeError(self.scheme)


class _Comparison():

    '''
    _Comparison class

    The constructor generates two lists: one of the schemes to be benchmarked, one for the benchmark instances.

    The function compare() runs the benchmarks 5 times for a specified number of algorithm iterations (default 1000) 
    and returns a list containing a dictionary of scheme and lowest benchmark pairs.
    
    The lowest benchmark is chosen because they represent the lower bound. The result in seconds is the time the 
    scheme requires to compute the algorithm result for the speficied number of times.
    '''

    def __init__(self, alg:str, scheme:str, *schemes):
        self.alg = alg
        self.scheme_list = [scheme]
        self.scheme_list.extend([arg for arg in schemes])
        self.bench_list = [_Benchmark(alg, self.scheme_list[i]) for i, _ in enumerate(self.scheme_list)]

    def compare(self, iterations)->tuple:
        try:
            compare_zip = zip([scheme for scheme in self.scheme_list],
                              [min(bench.benchmark(iterations))/iterations for bench in self.bench_list])
            return ((f'{self.alg} - average runtime in nanoseconds:',dict(compare_zip)))
        except (NameError, TypeError) as e:
            raise UnsupportedAlgorithmError(self.alg)


class _ComparisonAllSchemes(_Comparison):

    '''
    Convenience class for comparing all enabled schemes with each other.

    The constructor generates a list of all schemes and passes it to its parent class
    '''

    def __init__(self, alg:str):
        all_schemes = [scheme_ for scheme_ in oqs.get_enabled_kem_mechanisms()]
        super().__init__(alg, *all_schemes)


class _ComparisonCompleteScheme:

    '''Convenience class for comparing complete schemes with all three algorithms\
and their total sum of at least one scheme.'''

    def __init__(self, *scheme):
        self.keygen = _Comparison('KeyGeneration',*scheme)
        self.encap = _Comparison('Encapsulation',*scheme)
        self.decap = _Comparison('Decapsulation',*scheme)

    def compare_complete(self, iterations=1000):
        keygen_bench = self.keygen.compare(iterations)
        encap_bench = self.encap.compare(iterations)
        decap_bench = self.decap.compare(iterations)
        sum_list = []
        for i,_ in enumerate(keygen_bench[1]):
            sum_list.append(list(keygen_bench[1].values())[i] + list(encap_bench[1].values())[i] + list(decap_bench[1].values())[i])
        total_zip = zip(keygen_bench[1].keys(),sum_list)
        total_bench = ('Complete scheme - average runtime in nanoseconds:', dict(total_zip))
        return [keygen_bench, encap_bench, decap_bench, total_bench]

        
class _CompleteComparison(_ComparisonCompleteScheme):
    
    '''Convenience class for complete comparison of all schemes and all algorithms.'''
    
    def __init__(self):
        all_schemes = [scheme_ for scheme_ in oqs.get_enabled_kem_mechanisms()]
        super().__init__(*all_schemes)


def liboqs_bench(alg:str, *schemes, iterations:int=1000):

    '''Usage: liboqs_bench({'help'|'schemes'|algorithm} [,scheme1[,scheme2[,scheme3,...]]] [,iterations=int])

Small Python utility benchmarking PQC KEM algorithms and \
schemes from the liboqs-python library. The program measures each \
algorithm step separately for as many schemes as required by running \
the step a specified number of iterations for five trials in total. \
It then chooses the fastest trial and calculates the average.

Main function for interactive usage, calls classes according to arguments provided.

First argument is the algorithm step: "KeyGeneration", "Encapsulation", "Decapsulation", or "complete".

After the first argument, as many scheme names as desired can be provided as positional arguments or
the argument "all" in order to compare all schemes with each other.

Finally, the user can provide an integer as keyword argument "iterations" to specify the number of 
iterations each benchmark round should use. The default number is 1000.

For help, the user can pass as the first argument "help", which prints out this docstring, or "schemes", 
which returns a list of supported KEM schemes.
    '''
    
    match alg:
        case 'help':
            print(liboqs_bench.__doc__)
        
        case 'schemes':
            schemes_str = 'Supported KEM schemes:\n'
            for scheme_name in oqs.get_enabled_kem_mechanisms():
                schemes_str += f'\n"{scheme_name}"'
            print(schemes_str)     
        
        case 'complete':
            try:
                if schemes[0] == 'all':
                    return _CompleteComparison().compare_complete(iterations)
            except IndexError:
                raise NoSchemeProvidedError
            else:
                return _ComparisonCompleteScheme(*schemes).compare_complete(iterations)
        
        case _:
            try:
                if schemes[0] == 'all':
                    return [_ComparisonAllSchemes(alg).compare(iterations)]
            except IndexError:
                raise NoSchemeProvidedError        
            else:
                return [_Comparison(alg, *schemes).compare(iterations)]

Overwriting liboqs_bench.py


In [175]:
%%writefile argument_parser.py
'''Command line argument parser.'''

import logging
_logger = logging.getLogger('__main__.' + __name__)

from .__init__ import __version__
from oqs.oqs import get_enabled_kem_mechanisms as kem_schemes
import argparse

class ArgumentParser():
    
    '''
    Class for parsing command line arguments to be called by __main__.py.
    '''
    def __init__(self):
        global __version__
        alg_choices=['KeyGeneration', 'Encapsulation', 'Decapsulation', 'complete']
        scheme_choices=[scheme_name for scheme_name in kem_schemes()].append('all')
        
        self.parser = argparse.ArgumentParser(prog='liboqs_bench', 
                                              description='Small Python utility benchmarking PQC KEM algorithms and \
                                              schemes from the liboqs-python library. The program measures each \
                                              algorithm step separately for as many schemes as required by running \
                                              the step a specified number of iterations for five trials in total. \
                                              It then chooses the fastest trial and calculates the average.')
        self.parser.add_argument('alg',  metavar='algorithm', choices=alg_choices, nargs='?',
                            help=f'the desired algorithm step or "complete": {alg_choices}')
        self.parser.add_argument('scheme', nargs='*',
                            choices=[scheme_name for scheme_name in kem_schemes()].append('all'),
                           help=f'at least one KEM scheme or "all" - for full list use -l option')
        self.parser.add_argument('-i', '--iterations', type=int, default=1000, metavar='int', 
                                 help='set the number of iterations of each benchmark trial (default=1000)')
        self.parser.add_argument('-l', '--list-schemes', action='store_true', help='print the full list of supported KEM schemes')
        self.parser.add_argument('-d', '--debug', action='store_true', help='set log level to DEBUG and enable oqs warnings and logs (STUB FUNCTIONALITY)')
        self.parser.add_argument('-v', '--version', action='version', version=f'%(prog)s-{__version__}')

Overwriting argument_parser.py


In [176]:
%%writefile __main__.py
'''
Small Python utility benchmarking PQC KEM algorithms and \
schemes from the liboqs-python library. The program measures each \
algorithm step separately for as many schemes as required by running \
the step a specified number of iterations for five trials in total. \
It then chooses the fastest trial and calculates the average.

Main entry point for user interaction. 

Calls the argument parser for command line arguments, passes them to liboqs_bench, and prints the results.
'''
import logging
_logger = logging.getLogger(__name__)
_handler = logging.StreamHandler()
_formatter = logging.Formatter('%(name)s - %(levelname)s - %(message)s')
_handler.setFormatter(_formatter)
_logger.addHandler(_handler)

from oqs.oqs import get_enabled_kem_mechanisms as kem_schemes
from .liboqs_bench import liboqs_bench

from .argument_parser import ArgumentParser

argparser = ArgumentParser()
arguments = argparser.parser.parse_args()

if arguments.debug:
    logging.captureWarnings(False)
    logging.getLogger('oqs.oqs').disabled = False
    _logger.setLevel(logging.DEBUG)

if arguments.list_schemes:
    schemes_str = 'Supported KEM schemes:\n'
    for scheme_name in kem_schemes():
        schemes_str += f'\n"{scheme_name}"'
    print(schemes_str)
    
else:
    results = liboqs_bench(arguments.alg, *arguments.scheme, iterations=arguments.iterations)
    for outer in results:
        print(outer[0])
        for result in outer[1].items():
            print(result)

Overwriting __main__.py
